# Lag effect Analysis

### "정책 효과가 강하게 나타난 시점일수록, 모델은 더 쉽게 결과(Y)를 예측할 수 있다"
1. 정책이 효과를 발휘하면, Y(조직성과/개인결과)가 X(정책포함 정보)에 더 강하게 종속된다.
- 정책 도입 전에는 조직성과(Y)가 다양한 외적 요인에 따라 들쭉날쭉 (예측 어려움).
- 정책이 효과를 발휘하면 → 특정 정책 조건(X)에 따라 Y가 더 일관되고 예측 가능하게 변화함.
- 즉, 정책이 Y에 설명력을 부여한다.

2. 모델은 설명력 높은 관계일수록 예측 성능이 높다.
- 머신러닝 모델의 본질: 입력 X와 출력 Y 사이의 규칙성을 찾아내는 것.
- 따라서, 만약 정책 효과로 인해 X (정책포함) → Y의 인과 경로가 강해졌다면,
- 모델은 더 쉽게 일반화 가능한 패턴을 학습함.
- 결과적으로, 정확도, F1, AUC 등의 성능이 높아짐.

1. Stage 1
    - 2020년 데이터 X + 2021년 데이터 y => 모델-2020_2021
    - 2020년 데이터 X + 2023년 데이터 y => 모델-2020_2023
    => 각 모델의 퍼포먼스 측정 (강한 예측력을 보이는 모델 = 그게 결국 영향력)
    => 각 모델 해석력 지수 얻기 / 중요도에 따라서 인풋값의 가중치 다르게 줌


2. Stage 2
    - 2021년 데이터 X + 2021년 데이터 y => 모델-2021_2021
    - 2021년 데이터 X + 2023년 데이터 y => 모델-2022_2023

3. Stage 3
    - 2022년 데이터 X + 2023년 데이터 y => 모델-2022_2023

4. Stage 4
    - 2023년 데이터 X + 2023년 데이터 y => 모델-2023_2023

---

- Dataset: 2020 / 2021 / 2022 / 2023, 근데 2021 이랑 2023 데이터만 target label 갖고 있음
- Y : Target label
    - ~~2021 : C21C05_01H1 (8) 전체 숙련수준  / C21C05_01H2 (8) 전체 경쟁력~~
    - ~~2023 : C23C05_01H1 (8) 전체 숙련수준  / C23C05_01H2 (8) 전체 경쟁력~~ => 이렇게 하려했는데 C21C05_01H1의 Nan값이 88%
    - 2021 : C21C05_01H2 (8) 전체 경쟁력
    - 2023 : C23C05_01H2 (8) 전체 경쟁력

- X :

In [6]:
import matplotlib.pyplot as plt
import utils_, config, model
#from github.V1.utils_ import data_processing
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')


def get_all_data(year_list):
    file_names = config.file_names       # Expected file names (e.g., ['file1.csv', 'file2.csv'])
    file_list = os.listdir(config.path)  # All files in the directory
    dataset = {}                         # Final dictionary to store data
    year_cnt = -1                        # Counter to map years to files

    for expected_file in file_names:
        for actual_file in file_list:
            if expected_file == actual_file:
                year_cnt += 1
                current_year = year_list[year_cnt]
                print(f"{expected_file} ===> {current_year} data")

                # Load data
                df, meta = utils_.data_import(os.path.join(config.path, expected_file))

                # Store in dataset dict with year as key
                dataset[current_year] = {
                    'data': df,
                    'meta': meta
                }

    return dataset


def analyze_dataframe_step1(df: pd.DataFrame, meta, verbose):
    non_numeric_info = {}
    Meta_col = []
    total_rows = len(df)

    for idx, col in enumerate(df.columns):
        # 숫자 변환 불가능한 값 마스크
        non_numeric_mask = ~pd.to_numeric(df[col], errors='coerce').notna()
        non_numeric_count = non_numeric_mask.sum()
        nan_count = df[col].isna().sum()

        if non_numeric_count > 0:
            non_numeric_info[col] = {
                'non_numeric': non_numeric_count,
                'nan': nan_count
            }
            Meta_col.append(idx)

    # 결과 출력
    print("\n\n🔍 숫자가 아닌 character가 포함된 컬럼들 및 해당 row 수:")
    for idx, (col, stats) in enumerate(non_numeric_info.items()):
        label = meta.column_labels[Meta_col[idx]]
        if verbose:
            print(f"- {col} ({label}): 숫자 아님 {stats['non_numeric']}개 / NaN {stats['nan']}개")



def clean_dataframe_step1(df: pd.DataFrame, columns_to_drop: list, verbose):
    print("\n=================================\nPreprocessing\n=================================\n")
    total_rows = len(df)

    cols_to_drop = set(columns_to_drop)  # 삭제할 컬럼 집합

    for idx, col in enumerate(df.columns):
        nan_count = df[col].isna().sum()
        nan_ratio = nan_count / total_rows

        # NaN이 25% 미만이면 평균으로 대체
        if nan_count > 0 and nan_ratio < 0.25:
            try:
                mean_val = pd.to_numeric(df[col], errors='coerce').mean()
                df[col] = df[col].fillna(int(mean_val))
                if verbose:
                    print(f"→ {col}: NaN {nan_count}개 평균({int(mean_val):.2f})으로 대체 완료")
            except:
                if verbose:
                    print(f"→ {col}: 평균 계산 불가 (비숫자형 포함 등)")
                pass
        # NaN이 25% 이상이면 삭제 리스트에 추가
        elif nan_ratio >= 0.25:
            if verbose:
                print(f"→ {col}: NaN 비율 {nan_ratio:.2%}로 삭제 대상 추가")
            cols_to_drop.add(col)

    # 컬럼 삭제
    cleaned_df = df.drop(columns=list(cols_to_drop), errors='ignore')

    return cleaned_df


def target_variable_check(dataset, target_variable):
    for year in ['2021', '2023']:
        df, meta = dataset[year]['data'], dataset[year]['meta']
        #col_names = [f'C{year[2:]}C05_01H1', f'C{year[2:]}C05_01H2']
        #col_names = [f'C{year[2:]}C05_01H2']
        col_names = [f'C{year[2:]}{target_variable}']
        for col_name in col_names:
            nan_count = df[col_name].isna().sum()
            total = df.shape[0]
            print(f"{year} - {col_name} : NaN {nan_count}개 / 전체 {total}개 ({nan_count / total:.2%})")
            print(df[f'{col_name}'].value_counts())


def clean_target_classes(df: pd.DataFrame, target_col='target') -> pd.DataFrame:
    # 3인 행 삭제
    df = df[df[target_col] != 3].copy()

    # 1,2 -> 0
    df.loc[df[target_col].isin([1, 2]), target_col] = 0

    # 4,5 -> 1
    df.loc[df[target_col].isin([4, 5]), target_col] = 1

    return df.reset_index(drop=True), np.array(df[target_col])



In [2]:
year_list = config.year               # List of years (e.g., [2018, 2020, 2022])

dataset = get_all_data(year_list)
target_variable_check(dataset, target_variable='C05_01H2')

HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data
2021 - C21C05_01H2 : NaN 0개 / 전체 500개 (0.00%)
C21C05_01H2
3.0    386
4.0     72
2.0     35
1.0      4
5.0      3
Name: count, dtype: int64
2023 - C23C05_01H2 : NaN 0개 / 전체 500개 (0.00%)
C23C05_01H2
3.0    383
4.0     94
2.0     18
5.0      3
1.0      2
Name: count, dtype: int64


In [ ]:
#year_list = [2020]
# Q. 근데 각 데이터셋에서 col들이 다 동일해야되나? 예를 들어 특정 년도에 nan값이 많아서 특정 col을 삭제했으면 다른 데이터들도 삭제해야 되는거 아닌가



for year in ['2021']:
    #print(dataset[year]['data'].shape)
    #utils_.see_col_idx_and_name(dataset[year]['data'], dataset[year]['meta'])

    df, meta = dataset[year]['data'], dataset[year]['meta']
    analyze_dataframe_step1(df, meta, verbose=True)

    if year == '2021' or year == '2023':
        new_df, y = clean_target_classes(df, target_col=f'C{year[2:]}C05_01H2')
        X = clean_dataframe_step1(new_df, columns_to_drop=[f'C{year[2:]}_ID1', f'C{year[2:]}C05_01H2'], verbose=False)  #기업아이디, label 삭제
    else:
        if year == '2020':
            _, y = clean_target_classes(df, target_col=f'C{year[2:]}C05_01H2')



    X_train, X_test, y_train, y_test = utils_.data_aug_smote(X, y)
    #X_train, X_test, y_train, y_test = utils_.data_aug_smote_tomek(X, y)
    #X_train, X_test, y_train, y_test = utils_.data_undersample(X, y)
    print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
    model.XGBoost(X_train, X_test, y_train, y_test, col_name='Y-var')


In [12]:
dataset['2020']['data']

,C20_ID1,C20_IND1,C20_SCALE,C20_SCALE2,C20_KSIC1,C20_KSIC2,C20_KSIC3,C20_KSIC4,C20_TYPE,C20_SEX1,...,C20D04_03A2,C20D04_03B2,C20D04_03C2,C20D04_03A3,C20D04_03B3,C20D04_03C3,C20D04_031,C20D04_032,C20D04_033,C20D04_04
0,1.0,1.0,1.0,1.0,1.0,20.0,201.0,2012.0,1.0,1.0,...,95.0,NaN,NaN,1.0,NaN,NaN,3.0,NaN,NaN,2.0
1,2.0,1.0,2.0,2.0,1.0,11.0,112.0,1120.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
2,5.0,2.0,3.0,3.0,2.0,65.0,651.0,6511.0,3.0,1.0,...,84.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,3.0
3,9.0,3.0,3.0,3.0,4.0,72.0,721.0,7211.0,3.0,1.0,...,14.0,NaN,NaN,2.0,NaN,NaN,1.0,NaN,NaN,3.0
4,10.0,1.0,2.0,2.0,1.0,29.0,292.0,2921.0,1.0,1.0,...,87.0,NaN,NaN,2.0,NaN,NaN,1.0,NaN,NaN,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,4041.0,1.0,1.0,1.0,1.0,30.0,303.0,3039.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
506,4043.0,1.0,1.0,1.0,1.0,27.0,271.0,2719.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507,4052.0,1.0,1.0,1.0,1.0,10.0,107.0,1079.0,3.0,1.0,...,60.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,3.0
508,4061.0,1.0,1.0,1.0,1.0,27.0,273.0,2730.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
def compare_company_ids_in_dataset(dataset):
    years = list(dataset.keys())
    id_sets = {}

    for year in years:
        col_name = f"C{year[2:]}_ID1"
        df = dataset[year]['data']
        id_sets[year] = set(df[col_name].dropna().unique())

    for i in range(len(years)):
        for j in range(i+1, len(years)):
            y1, y2 = years[i], years[j]
            common = id_sets[y1].intersection(id_sets[y2])
            print(f"{y1} & {y2} 교집합 기업 수: {len(common)}")

    return id_sets

_ = compare_company_ids_in_dataset(dataset)

2020 & 2021 교집합 기업 수: 455
2020 & 2022 교집합 기업 수: 439
2020 & 2023 교집합 기업 수: 410
2021 & 2022 교집합 기업 수: 471
2021 & 2023 교집합 기업 수: 428
2022 & 2023 교집합 기업 수: 446
